# AI Repair Agent

## Objective

This notebook analyzes quarantined records generated by the Silver layer using a Large Language Model (LLM).

Responsibilities:

- Read quarantined datasets
- Analyze unresolved anomalies
- Generate repair recommendations
- Assign confidence scores
- Create AI-approved dataset
- Merge approved records with Silver

## Step 1 - Import Libraries

In [0]:
import os
import json
import pandas as pd

from pyspark.sql.functions import *
from pyspark.sql.types import *

## Step 2 - Load Configuration

In [0]:
from utils.config import *

## Step 3 - Load Latest Silver Batch

In [0]:
silver_batches = sorted(os.listdir(SILVER_PATH))

LATEST_BATCH = silver_batches[-1]

print("Latest Silver Batch:")
print(LATEST_BATCH)

In [0]:
LATEST_SILVER_PATH = os.path.join(
    SILVER_PATH,
    LATEST_BATCH
)

print(LATEST_SILVER_PATH)

## Step 4 - Load Silver and Quarantine Datasets

In [0]:
bus_silver_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "bus_gps")
)

emergency_silver_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "emergency")
)

bus_quarantine_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "bus_quarantine")
)

emergency_quarantine_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "emergency_quarantine")
)

In [0]:
print("="*60)
print("Bus Silver :", bus_silver_df.count())
print("Emergency Silver :", emergency_silver_df.count())
print("Bus Quarantine :", bus_quarantine_df.count())
print("Emergency Quarantine :", emergency_quarantine_df.count())

##Install AI SDKs

In [0]:
%pip install -q google-genai
%pip install -q groq


## Initialize AI Providers

In [0]:
from google import genai
from gemini_config import GEMINI_API_KEY

In [0]:
client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("✅ Gemini initialized successfully.")

In [0]:
from groq import Groq
from groq_config import GROQ_API_KEY

In [0]:
groq_client = Groq(
    api_key=GROQ_API_KEY
)

print("✅ Groq initialized successfully.")

## AI Provider Router

In [0]:
import json

def analyze_record(prompt):

    gemini_error = None

    # -----------------------
    # Try Gemini First
    # -----------------------
    try:

        response = client.models.generate_content(
            model="gemini-flash-latest",
            contents=prompt
        )

        return json.loads(response.text)

    except Exception as e:

        gemini_error = str(e)
        print("⚠️ Gemini unavailable. Switching to Groq...")

    # -----------------------
    # Try Groq
    # -----------------------
    try:

        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return json.loads(
            response.choices[0].message.content
        )

    except Exception as groq_error:

        return {

            "recommendation": None,

            "confidence": 0,

            "reason": f"Gemini Error: {gemini_error} | Groq Error: {str(groq_error)}",

            "action": "ERROR"

        }

#Test AI Router

In [0]:
result = analyze_record(
    "Reply ONLY with this JSON: {\"recommendation\":\"READY\",\"confidence\":100,\"reason\":\"Test\",\"action\":\"AUTO_APPROVE\"}"
)

print(result)

## AI Prompt Template

In [0]:
def build_prompt(record):

    return f"""
You are an AI Data Quality Engineer for a Smart City platform.

Your responsibility is to infer the MOST LIKELY missing value using the available information.

Record:

Incident ID: {record["incident_id"]}
Zone: {record["zone"]}
Incident Type: {record["incident_type"]}
Severity: {record["severity"]}
Response Time: {record["response_time"]}
Status: {record["status"]}

Task:

1. Infer the most likely Zone.
2. Do NOT reply "I don't know" unless there is absolutely no reasonable inference.
3. Assign a confidence score between 0 and 100.
4. If confidence >= 90:
   action = AUTO_APPROVE
5. Otherwise:
   action = HUMAN_REVIEW

Return ONLY valid JSON.

Example:

{{
    "recommendation":"Zone B",
    "confidence":93,
    "reason":"Medical incidents with similar characteristics are commonly observed in Zone B.",
    "action":"AUTO_APPROVE"
}}
"""

## AI Batch Processing

In [0]:
from datetime import datetime
import json

ai_audit_records = []

quarantine_records = (
    emergency_quarantine_df
    .limit(10)
    .toPandas()
    .to_dict("records")
)

print("Total Quarantine Records :", len(quarantine_records))
#emergency_quarantine_df.toPandas().to_dict("records") this is for all records but due to gemini limitation testing on 10

print("Total Quarantine Records :", len(quarantine_records))

In [0]:
for record in quarantine_records:

    try:

        # Build Prompt
        prompt = build_prompt(record)

        # Analyze using AI Router (Gemini -> Groq Fallback)
        result = analyze_record(prompt)

        # Store AI Audit
        ai_audit_records.append({

            "incident_id": record["incident_id"],

            "recommendation": result["recommendation"],

            "confidence": result["confidence"],

            "action": (
                "AUTO_APPROVE"
                if result["confidence"] >= 60
                else "HUMAN_REVIEW"
            ),

            "reason": result["reason"],

            "processed_at": datetime.now(),

            "model": "Gemini/Groq Router"

        })

    except Exception as e:

        ai_audit_records.append({

            "incident_id": record["incident_id"],

            "recommendation": None,

            "confidence": 0,

            "action": "ERROR",

            "reason": str(e),

            "processed_at": datetime.now(),

            "model": "Gemini/Groq Router"

        })

## Create AI Audit DataFrame

In [0]:
print("AI Records Processed :", len(ai_audit_records))

In [0]:
ai_audit_df = pd.DataFrame(ai_audit_records)
ai_audit_df.head()

In [0]:
ai_audit_records[0]

##AI Decision Summary

In [0]:
print("=" * 60)
print("AI Decision Summary")
print("=" * 60)

print("AI Approved :", len(ai_audit_df[ai_audit_df["action"] == "AUTO_APPROVE"]))
print("Human Review :", len(ai_audit_df[ai_audit_df["action"] == "HUMAN_REVIEW"]))

##error check

In [0]:
print("Records in AI Audit:", len(ai_audit_df))

In [0]:
ai_audit_df[["incident_id", "action", "confidence"]]

In [0]:
ai_audit_df["action"].value_counts()

In [0]:
ai_audit_df[["incident_id", "reason"]]

In [0]:
for i, row in ai_audit_df.iterrows():
    print("=" * 80)
    print(row["incident_id"])
    print(row["reason"])